# Custom Transformers

## Overview

Custom sklearn-compatible transformers extend the pipeline system with domain-specific logic. By subclassing `BaseEstimator` and `TransformerMixin`, custom steps plug into `Pipeline`, `ColumnTransformer`, and `GridSearchCV` without modification.

**Two patterns:**

| Pattern | When |
|---|---|
| `FunctionTransformer` | Stateless transforms (log, ratio features) |
| `BaseEstimator + TransformerMixin` | Stateful transforms that need `.fit()` (e.g. outlier clipping fitted to training data, target-aware encoders) |

**The sklearn transformer contract:**
- `fit(X, y=None)`: learn from training data, return `self`
- `transform(X)`: apply learned transform, return array or DataFrame
- `fit_transform(X, y)`: convenience method (provided by `TransformerMixin`)
- `get_params()` / `set_params()`: enables `GridSearchCV` (provided by `BaseEstimator`)

---

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.utils.validation import check_is_fitted

rng = np.random.default_rng(42)
n = 400
df = pd.DataFrame({
    'elevation':   rng.uniform(50, 400, n),
    'nitrate':     rng.gamma(2, 2, n),
    'phosphorus':  rng.gamma(1.5, 1.5, n),
    'flow_rate':   rng.lognormal(1.5, 0.8, n),
    'catchment':   rng.choice(['North','South','East','West'], n),
})
df['richness'] = (25 - 0.04*df['elevation'] - 0.8*df['nitrate']
                  + rng.normal(0, 3, n)).clip(0)
X = df.drop('richness', axis=1)
y = df['richness']
print(f"Shape: {X.shape}")

Shape: (400, 5)


---
## Stateless Transform: FunctionTransformer

In [2]:
# Stateless: log1p transform -- no training-data statistics needed
log_transformer = FunctionTransformer(np.log1p, validate=True)

# Stateless: ratio feature engineering
def add_ratios(X):
    X = pd.DataFrame(X, columns=['elevation','nitrate','phosphorus','flow_rate'])
    X['N_P_ratio']     = X['nitrate'] / X['phosphorus'].clip(0.01)
    X['nutrient_load'] = X['nitrate'] * np.log1p(X['flow_rate'])
    return X.values

ratio_transformer = FunctionTransformer(add_ratios, validate=False)

# Test
X_num = X[['elevation','nitrate','phosphorus','flow_rate']].values
print("Before:", X_num[:2].round(3))
print("Log1p: ", log_transformer.transform(X_num[:2]).round(3))
print("Ratios:", ratio_transformer.transform(X_num[:2]).round(3))

Before: [[320.885   4.602   1.531   5.091]
 [203.607   4.099   3.942   4.382]]
Log1p:  [[5.774 1.723 0.929 1.807]
 [5.321 1.629 1.598 1.683]]
Ratios: [[320.885   4.602   1.531   5.091   3.005   8.315]
 [203.607   4.099   3.942   4.382   1.04    6.899]]


---
## Stateful Custom Transformer

In [3]:
class Winsorizer(BaseEstimator, TransformerMixin):
    """Clip values to [lower_q, upper_q] percentiles fitted on training data."""
    def __init__(self, lower_q=0.01, upper_q=0.99):
        self.lower_q = lower_q
        self.upper_q = upper_q

    def fit(self, X, y=None):
        X = np.array(X)
        self.lower_ = np.nanpercentile(X, self.lower_q*100, axis=0)
        self.upper_ = np.nanpercentile(X, self.upper_q*100, axis=0)
        return self

    def transform(self, X):
        check_is_fitted(self)   # raises if fit() not called
        X = np.array(X, dtype=float)
        return np.clip(X, self.lower_, self.upper_)

# Inject outliers
X_out = X[['elevation','nitrate','phosphorus','flow_rate']].copy()
X_out.iloc[0, 1] = 150.0   # extreme nitrate
X_out.iloc[1, 3] = 2000.0  # extreme flow

wins = WinsorSerializer = WinsorEncoder = WinsorEncoder = Winsorizer(lower_q=0.01, upper_q=0.99)
X_wins = wins.fit_transform(X_out.values)
print(f"Original max nitrate: {X_out['nitrate'].max():.1f}")
print(f"Winsorised max:       {X_wins[:,1].max():.3f}")
print(f"Clip bounds fitted from training data: {wins.lower_[1]:.3f} – {wins.upper_[1]:.3f}")

Original max nitrate: 150.0
Winsorised max:       11.960
Clip bounds fitted from training data: 0.278 – 11.960


---
## Integrating Custom Transformers into a Pipeline

In [4]:
class DomainRatioFeatures(BaseEstimator, TransformerMixin):
    """Add ecologically-motivated ratio and interaction features."""
    def __init__(self, add_load=True, clip_min=0.01):
        self.add_load = add_load
        self.clip_min = clip_min

    def fit(self, X, y=None):
        self.feature_names_in_ = ['elevation','nitrate','phosphorus','flow_rate']
        return self

    def transform(self, X):
        check_is_fitted(self)
        df = pd.DataFrame(X, columns=self.feature_names_in_)
        df['N_P_ratio'] = df['nitrate'] / df['phosphorus'].clip(self.clip_min)
        if self.add_load:
            df['nutrient_load'] = df['nitrate'] * np.log1p(df['flow_rate'])
        return df.values

num_cols = ['elevation','nitrate','phosphorus','flow_rate']
cat_cols = ['catchment']
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

pipe = Pipeline([
    ('wins',  Winsorizer(lower_q=0.02, upper_q=0.98)),
    ('ratios', DomainRatioFeatures(add_load=True)),
    ('scaler', StandardScaler()),
    ('reg',    GradientBoostingRegressor(n_estimators=200, random_state=42)),
])
# Numeric only for this demo
X_num_only = X[num_cols]
scores = cross_val_score(pipe, X_num_only, y, cv=5, scoring='r2')
print(f"CV R2 with custom transformer pipeline: {scores.mean():.3f} +/- {scores.std():.3f}")

CV R2 with custom transformer pipeline: 0.540 +/- 0.094


In [5]:
# GridSearchCV works seamlessly -- custom params accessible via __
param_grid = {
    'wins__upper_q':       [0.95, 0.99],
    'ratios__add_load':    [True, False],
    'ratios__clip_min':    [0.001, 0.01],
    'reg__n_estimators':   [100, 200],
}
gs = GridSearchCV(pipe, param_grid, cv=5, scoring='r2', n_jobs=-1)
gs.fit(X_num_only, y)
print(f"Best params: {gs.best_params_}")
print(f"Best CV R2:  {gs.best_score_:.3f}")
print("\nCustom transformer parameters are tunable via GridSearchCV")
print("because BaseEstimator provides get_params() / set_params()")

Best params: {'ratios__add_load': False, 'ratios__clip_min': 0.001, 'reg__n_estimators': 100, 'wins__upper_q': 0.99}
Best CV R2:  0.585

Custom transformer parameters are tunable via GridSearchCV
because BaseEstimator provides get_params() / set_params()


---

## Common Pitfalls

**1. Storing training-data statistics in `transform()` instead of `fit()`**  
If percentiles, means, or other statistics are computed inside `transform()`, they will be recomputed on every call — including on test data — producing leakage. All statistics that depend on training data must be computed in `fit()` and stored as attributes ending with `_` (the sklearn convention for fitted attributes).

**2. Not calling `check_is_fitted()` at the start of `transform()`**  
Without this check, calling `transform()` before `fit()` produces a cryptic AttributeError when the fitted attribute is missing. `check_is_fitted(self)` raises a clear `NotFittedError` immediately.

**3. Modifying the input array in-place inside `transform()`**  
If `transform()` modifies `X` directly (e.g. `X[:,0] = ...`), it modifies the caller's data. Always copy: `X = np.array(X, dtype=float)` or `X = X.copy()` at the start of `transform()`.

**4. Forgetting to return `self` from `fit()`**  
Python method chaining (`transformer.fit(X).transform(X)`) and `Pipeline` both require `fit()` to return `self`. A missing `return self` causes an `AttributeError` when the pipeline tries to chain the call.

**5. Not declaring all constructor parameters as attributes with the same name**  
`BaseEstimator.get_params()` uses introspection to find parameters — it looks for attributes whose names match the `__init__` argument names exactly. If `__init__` accepts `lower_q` but stores it as `self.low`, `get_params()` breaks and `GridSearchCV` cannot tune that parameter.

---
*python_methods_library - Samantha McGarrigle*